# 25. 사업보고서 원문 추출 비교

**목적:** 공시 목록의 `rcept_no`로 실제 사업보고서 원문을 받은 뒤, 단순 평탄화(A)와 제목·표 경계를 표시하는 추출(B)이 검색 가능한 텍스트를 어떻게 만드는지 확인합니다. 원문 구조를 모른 채 chunking·embedding을 비교하면 검색 실패의 원인을 구분하기 어렵습니다.

**범위:** 삼성전자·리노공업의 2024 사업연도 사업보고서 최종 제출본 각 1건. 이는 *2025년 접수일*에 제출된 2024 사업연도 보고서입니다. 다른 기업·결산월까지 일반화하지 않습니다.

**이번에 판단하지 않는 것:** 파서 최종 채택, chunking·embedding·retrieval 성능, LLM 답변 품질. 정답 근거 문단을 독립적으로 표시하기 전에는 Recall@K·MRR을 계산하지 않습니다. API 키와 원문 전체를 출력하지 않습니다.

참고: [OpenDART 공시검색](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS001&apiId=2019001), [공시서류원본파일](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS001&apiId=2019003), [LlamaIndex Node Parser](https://developers.llamaindex.ai/python/framework/module_guides/loading/node_parsers/).

## 1. 환경과 입력 범위

In [1]:
from collections import Counter
from io import BytesIO
from pathlib import Path
from time import perf_counter
import os
import re
import sys
import zipfile

from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from dotenv import load_dotenv
import pandas as pd
import requests

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / '.env').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 최상위 .env를 찾지 못했습니다.')
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env에 OPENDART_API_KEY가 필요합니다.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from dart.client import find_corp_info
print('실행 환경과 API 키 존재 여부 확인 완료 (키 값은 출력하지 않음)')

실행 환경과 API 키 존재 여부 확인 완료 (키 값은 출력하지 않음)


In [2]:
COMPANIES = ['삼성전자', '리노공업']
BUSINESS_YEAR = 2024
RECEIPT_START, RECEIPT_END = '20250101', '20251231'
LIST_URL = 'https://opendart.fss.or.kr/api/list.json'
DOCUMENT_URL = 'https://opendart.fss.or.kr/api/document.xml'
print('회사:', COMPANIES, '| 사업연도:', BUSINESS_YEAR, '| 접수일:', RECEIPT_START, '~', RECEIPT_END)

회사: ['삼성전자', '리노공업'] | 사업연도: 2024 | 접수일: 20250101 ~ 20251231


## 2. 사업보고서 접수번호 선택

`pblntf_detail_ty=A001`은 사업보고서 유형입니다. 목록 결과에서 사업연도 표기까지 확인하고, 여러 최종 제출본이 있으면 이 실험에서는 가장 최근 접수번호를 사용합니다. 선택 후보를 먼저 출력하므로 잘못된 문서를 조용히 고르지 않습니다.

In [3]:
def get_json(url, params):
    try:
        response = requests.get(url, params=params, timeout=40)
        if response.status_code != 200:
            raise RuntimeError(f'OpenDART HTTP {response.status_code}')
        return response.json()
    except requests.RequestException:
        raise RuntimeError('OpenDART 요청 실패: 연결 또는 시간 제한을 확인하세요.') from None

def list_annual_reports(corp_code):
    params = {'crtfc_key': API_KEY, 'corp_code': corp_code, 'bgn_de': RECEIPT_START,
              'end_de': RECEIPT_END, 'pblntf_detail_ty': 'A001',
              'pblntf_ty': 'A', 'last_reprt_at': 'Y', 'page_count': 100}
    rows = []
    page_no = 1
    expected_total = expected_pages = None
    while True:
        payload = get_json(LIST_URL, {**params, 'page_no': page_no})
        if page_no == 1 and payload.get('status') == '013':
            return []
        if payload.get('status') != '000':
            raise RuntimeError(f"공시 목록 오류: {payload.get('status')} / {payload.get('message')}")
        try:
            total, pages, current = (int(payload[key]) for key in ('total_count', 'total_page', 'page_no'))
        except (KeyError, TypeError, ValueError):
            raise RuntimeError('공시 목록 페이지 메타데이터가 비정상입니다.') from None
        page_rows = payload.get('list')
        if pages < 1 or current != page_no or not isinstance(page_rows, list) or any(not isinstance(row, dict) for row in page_rows):
            raise RuntimeError('공시 목록 페이지 번호 또는 list 형식이 비정상입니다.')
        if expected_total is None:
            expected_total, expected_pages = total, pages
        elif (total, pages) != (expected_total, expected_pages):
            raise RuntimeError('공시 목록 조회 도중 전체 건수 또는 페이지 수가 변경됐습니다.')
        rows.extend(page_rows)
        if page_no >= expected_pages:
            receipts = [str(row.get('rcept_no') or '') for row in rows]
            if any(len(number) != 14 or not number.isdigit() for number in receipts):
                raise RuntimeError('공시 목록에 14자리 숫자가 아닌 접수번호가 있습니다.')
            if len(rows) != expected_total or len(set(receipts)) != len(rows):
                raise RuntimeError('공시 목록의 수집 건수·접수번호가 API 총수와 다릅니다.')
            return rows
        page_no += 1

In [4]:
report_candidates = []
for company in COMPANIES:
    corp_code = find_corp_info(company, API_KEY)['corp_code']
    for row in list_annual_reports(corp_code):
        if re.search(rf'\b{BUSINESS_YEAR}\.12\b', row.get('report_nm', '')):
            report_candidates.append({'company': company, 'corp_code': corp_code,
                                      'rcept_no': row['rcept_no'], 'rcept_dt': row['rcept_dt'],
                                      'report_nm': row['report_nm']})
if not report_candidates:
    raise RuntimeError('2024.12 사업보고서 후보가 없습니다. 날짜와 보고서명을 직접 확인하세요.')
display(pd.DataFrame(report_candidates).sort_values(['company', 'rcept_no']))

,company,corp_code,rcept_no,rcept_dt,report_nm
1,리노공업,00369657,20250318000229,20250318,사업보고서 (2024.12)
0,삼성전자,00126380,20250311001085,20250311,사업보고서 (2024.12)


In [5]:
selected_reports = []
for company in COMPANIES:
    matches = [row for row in report_candidates if row['company'] == company]
    if not matches:
        raise RuntimeError(f'{company}: 해당 사업연도 보고서 후보가 없습니다.')
    selected_reports.append(max(matches, key=lambda row: row['rcept_no']))
display(pd.DataFrame(selected_reports))
print('위 접수번호가 의도한 사업보고서인지 확인한 뒤 다음 셀을 실행하세요.')

,company,corp_code,rcept_no,rcept_dt,report_nm
0,삼성전자,00126380,20250311001085,20250311,사업보고서 (2024.12)
1,리노공업,00369657,20250318000229,20250318,사업보고서 (2024.12)


위 접수번호가 의도한 사업보고서인지 확인한 뒤 다음 셀을 실행하세요.


## 3. 원문 ZIP 확인

공시검색은 문서 **목록**만 반환합니다. 아래에서 선택한 `rcept_no`로 원문 ZIP을 별도 호출합니다. ZIP 안 XML 파일명과 크기를 출력해 파싱 입력을 확인합니다. 원문 본문 전체는 출력하거나 로컬에 저장하지 않습니다.

In [6]:
def download_document_xml(rcept_no):
    start = perf_counter()
    try:
        with requests.get(DOCUMENT_URL, params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                          timeout=120, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f'원문 API HTTP {response.status_code}')
            chunks, received = [], 0
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                received += len(chunk)
                if received > 60_000_000:
                    raise RuntimeError('원문 ZIP이 60MB를 넘어 다운로드를 중단합니다.')
                chunks.append(chunk)
    except requests.RequestException:
        raise RuntimeError('원문 API 요청 실패: 연결 또는 시간 제한을 확인하세요.') from None
    payload = b''.join(chunks)
    if not zipfile.is_zipfile(BytesIO(payload)):
        raise RuntimeError(f'{rcept_no}: 원문 응답이 ZIP이 아닙니다. API 상태와 접수번호를 확인하세요.')
    with zipfile.ZipFile(BytesIO(payload)) as archive:
        xml_entries = [info for info in archive.infolist() if info.filename.lower().endswith('.xml')]
        if not xml_entries:
            raise RuntimeError(f'{rcept_no}: ZIP 안에 XML 파일이 없습니다.')
        chosen = max(xml_entries, key=lambda info: info.file_size)
        if chosen.file_size > 60_000_000 or sum(info.file_size for info in archive.infolist()) > 120_000_000:
            raise RuntimeError(f'{rcept_no}: 압축 해제 크기가 실험 한도를 넘습니다.')
        xml_bytes = archive.read(chosen)
        entries = [{'filename': info.filename, 'bytes': info.file_size}
                   for info in archive.infolist() if not info.is_dir()]
    return xml_bytes, chosen.filename, entries, perf_counter() - start

## 4. 원문 구조와 A/B 추출 비교

A는 XML의 텍스트를 한 줄로 이어 붙입니다. B는 같은 텍스트를 사용하되 제목과 표 셀의 경계를 표시합니다. 두 방식 모두 원문의 정확한 문단 구조를 보장하지 않으므로, 출력 예시를 직접 읽고 손실·중복·잡음 여부를 확인합니다.

In [7]:
def inspect_xml(xml_bytes):
    soup = BeautifulSoup(xml_bytes, 'lxml-xml')
    if soup.find() is None:
        raise RuntimeError('XML에서 태그를 찾지 못했습니다.')
    tag_counts = Counter(tag.name.upper() for tag in soup.find_all(True))
    title_samples = [tag.get_text(' ', strip=True)[:160] for tag in soup.find_all(['TITLE', 'SUBTITLE'])]
    title_samples = [title for title in title_samples if title]
    return soup, tag_counts, title_samples

In [8]:
def extract_a(soup):
    return re.sub(r'\s+', ' ', soup.get_text(' ', strip=True)).strip()

def extract_b(soup):
    parts = []
    for node in soup.descendants:
        if isinstance(node, Tag) and node.name.upper() == 'TR':
            parts.append('\n')
        if not isinstance(node, NavigableString) or isinstance(node, Comment):
            continue
        value = re.sub(r'\s+', ' ', str(node)).strip()
        if not value:
            continue
        ancestors = {tag.name.upper() for tag in node.parents if tag.name}
        if ancestors & {'TITLE', 'SUBTITLE'}:
            parts.append('\n## ' + value + '\n')
        elif ancestors & {'TD', 'TH'}:
            parts.append(value + ' | ')
        else:
            parts.append(value + ' ')
    return re.sub(r' *\n *', '\n', ''.join(parts)).strip()

In [9]:
documents, zip_rows, extraction_rows = {}, [], []
for report in selected_reports:
    company = report['company']
    xml_bytes, filename, entries, seconds = download_document_xml(report['rcept_no'])
    soup, counts, titles = inspect_xml(xml_bytes)
    document = {**report, 'xml_filename': filename}
    zip_rows.append({'company': company, 'rcept_no': report['rcept_no'],
                     'zip_entries': len(entries), 'chosen_xml': filename,
                     'xml_bytes': len(xml_bytes), 'download_seconds': round(seconds, 2)})
    print(company, '| ZIP 파일:', entries[:10], '...' if len(entries) > 10 else '')
    print(company, '| 상위 태그:', counts.most_common(12), '| 제목 예시:', titles[:12])
    if not titles:
        print('주의: TITLE/SUBTITLE 태그를 찾지 못했습니다. 원문 구조를 직접 확인하세요.')
    for variant, extractor in [('A_flat', extract_a), ('B_boundaries', extract_b)]:
        start = perf_counter()
        value = extractor(soup)
        if not value:
            raise RuntimeError(f'{company} {variant}: 추출된 텍스트가 없습니다.')
        if variant == 'B_boundaries' and '## ' not in value:
            print(f'주의: {company} B에서 제목 경계가 표시되지 않았습니다.')
        document[variant] = value
        extraction_rows.append({'company': company, 'variant': variant,
                                'characters': len(value), 'lines': value.count('\n') + 1,
                                'marked_headings': value.count('\n## '),
                                'extract_seconds': round(perf_counter() - start, 2)})
    documents[company] = document
    del soup, xml_bytes
display(pd.DataFrame(zip_rows))
display(pd.DataFrame(extraction_rows))

삼성전자 | ZIP 파일: [{'filename': '20250311001085_00761.xml', 'bytes': 702251}, {'filename': '20250311001085_00760.xml', 'bytes': 596561}, {'filename': '20250311001085.xml', 'bytes': 6298681}] 
삼성전자 | 상위 태그: [('TD', 1426), ('TR', 557), ('TE', 353), ('COL', 333), ('P', 299), ('TH', 234), ('SPAN', 167), ('TABLE', 110), ('COLGROUP', 110), ('TBODY', 110), ('TU', 90), ('PGBRK', 53)] | 제목 예시: ['목     차', '【 대표이사 등의 확인 】', 'I. 회사의 개요', '1. 회사의 개요', '2. 회사의 연혁', '3. 자본금 변동사항', '4. 주식의 총수 등', '5. 정관에 관한 사항', 'II. 사업의 내용', '1. 사업의 개요', '2. 주요 제품 및 서비스', '3. 원재료 및 생산설비']
리노공업 | ZIP 파일: [{'filename': '20250318000229.xml', 'bytes': 1559021}, {'filename': '20250318000229_00760.xml', 'bytes': 414127}] 
리노공업 | 상위 태그: [('P', 4315), ('TE', 4126), ('TD', 3055), ('TR', 2918), ('COL', 2304), ('TH', 1983), ('TABLE', 900), ('COLGROUP', 900), ('TBODY', 900), ('THEAD', 338), ('TU', 239), ('SPAN', 139)] | 제목 예시: ['목     차', '【 대표이사 등의 확인 】', 'I. 회사의 개요', '1. 회사의 개요', '2. 회사의 연혁', '3. 자본금 변동사항', '4. 주식의 총수 등', '5. 정관

,company,rcept_no,zip_entries,chosen_xml,xml_bytes,download_seconds
0,삼성전자,20250311001085,3,20250311001085.xml,6298681,0.24
1,리노공업,20250318000229,2,20250318000229.xml,1559021,0.14


,company,variant,characters,lines,marked_headings,extract_seconds
0,삼성전자,A_flat,41453,1,0,0.00
1,삼성전자,B_boundaries,44808,590,16,0.03
2,리노공업,A_flat,149973,1,0,0.02
3,리노공업,B_boundaries,159565,3115,98,0.06


In [10]:
def context_window(text, keyword, radius=250):
    position = text.find(keyword)
    if position < 0:
        return '키워드를 추출 텍스트에서 찾지 못함'
    return text[max(0, position - radius):position + len(keyword) + radius]

for company, document in documents.items():
    print('\n###', company, '| 접수번호', document['rcept_no'])
    for keyword in ['사업의 내용', '위험', '주요 제품']:
        print('\n키워드:', keyword)
        for variant in ['A_flat', 'B_boundaries']:
            print(variant, '=>', context_window(document[variant], keyword))


### 삼성전자 | 접수번호 20250311001085

키워드: 사업의 내용
A_flat => 립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주소, 전화번호 및 홈페이지 주소: 경기도 수원시 영통구 삼성로 129(매탄동) 전화번호: 031-200-1114 홈페이지: https://www.samsung.com/sec 라. 주요 사업의 내용 당사는 제품의 특성에 따라 DX(Device eXperience), DS(Device Solutions) 2개의 부문과 패널 사업을 영위하는 SDC(삼성디스플레이㈜ 및 그 종속기업), 전장부품사업 등을 영위하는 Harman(Harman International Industries, Inc. 및 그 종속기업)으로 나누어 경영을 하고 있습니다. 당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄, DS 부문 산하 해외 5개 지역총괄
B_boundaries => 립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주소, 전화번호 및 홈페이지 주소: 경기도 수원시 영통구 삼성로 129(매탄동) 전화번호: 031-200-1114 홈페이지: https://www.samsung.com/sec 라. 주요 사업의 내용 당사는 제품의 특성에 따라 DX(Device eXperience), DS(Device Solutions) 2개의 부문과 패널 사업을 영위하는 SDC(삼성디스플레이㈜ 및 그 종속기업), 전장부품사업 등을 영위하는 Harman(Harman International Industries, Inc. 및 

## 해석할 때 볼 것

1. 선택된 접수번호가 정말 2024 사업연도 사업보고서인지 확인합니다. `rcept_dt`는 접수일이지 사업연도가 아닙니다.
2. ZIP에 XML이 여러 개라면 가장 큰 XML을 고른 현재 **실험용 규칙**이 본문을 가리키는지 확인합니다. 아닐 경우 여기서 중단하고 문서 선택 규칙을 다시 정해야 합니다.
3. A/B 출력에서 제목, 본문, 표 수치가 누락·중복·깨짐 없이 읽히는지 확인합니다. `characters`나 제목 개수가 많다고 검색 품질이 좋다는 뜻은 아닙니다.
4. 다음 평가에서는 사람이 확인한 원문 근거 문단과 질문을 정답으로 고정한 뒤, 파싱·chunking·retrieval 후보의 Recall@K, MRR, nDCG@K를 비교합니다. 이번 노트북만으로 파서를 채택하지 않습니다.

## 5. XML → B 표본 문자열 대조

앞에서는 추출된 글자 수와 예시를 봤습니다. 여기서는 각 회사의 XML에서 제목·일반 문단·표 행을 하나씩 골라, 그 안의 글자가 B 결과에 같은 순서로 남았는지 확인합니다. 앞 셀에서 XML을 메모리에서 지웠으므로 선택된 접수번호의 XML을 두 건 다시 내려받습니다.

비교할 때 B가 추가한 `##`와 표 구분자 `|`, 공백만 제거합니다. 원문에 실제 `|`가 들어간 구간은 잘못된 판정을 피하려고 표본에서 제외합니다. 이 검사는 선택한 6개 표본의 글자가 B 결과에 있는지 보는 점검입니다. 같은 XML 파서를 사용하고 반복 문구의 위치는 구분하지 못하므로 원본 XML 전체의 완전성이나 이후 RAG 검색 품질을 증명하지 않습니다.

In [ ]:
def comparison_text(value, from_b=False):
    if from_b:
        value = re.sub(r'(?m)^##\s*', '', value).replace('|', '')
    return re.sub(r'\s+', '', value)

def choose_source_samples(soup):
    headings = [tag for tag in soup.find_all(['TITLE', 'SUBTITLE'])
                if '사업의 내용' in tag.get_text(' ', strip=True)
                and '|' not in tag.get_text(' ', strip=True)]
    paragraphs = [tag for tag in soup.find_all('P')
                  if tag.find_parent('TABLE') is None
                  and 60 <= len(tag.get_text(' ', strip=True)) <= 500
                  and '|' not in tag.get_text(' ', strip=True)]
    table_rows = [tag for tag in soup.find_all('TR')
                  if len(tag.find_all(['TD', 'TH'])) >= 2
                  and 20 <= len(tag.get_text(' ', strip=True)) <= 300
                  and '|' not in tag.get_text(' ', strip=True)]
    if not headings or not paragraphs or not table_rows:
        raise RuntimeError('제목·일반 문단·표 행 중 표본을 찾지 못했습니다. XML 구조를 직접 확인하세요.')
    return {'제목': headings[-1], '일반 문단': paragraphs[len(paragraphs) // 2],
            '표 행': table_rows[len(table_rows) // 2]}

In [ ]:
sample_rows, integrity_rows = [], []
for company, document in documents.items():
    xml_bytes, filename, _, _ = download_document_xml(document['rcept_no'])
    if filename != document['xml_filename']:
        raise RuntimeError(f'{company}: 앞에서 선택한 XML 파일과 다릅니다.')
    soup, _, _ = inspect_xml(xml_bytes)
    full_b = document['B_boundaries']
    full_b_compact = comparison_text(full_b, from_b=True)
    source_text = soup.get_text(' ', strip=True)
    integrity_rows.append({'company': company, 'source_replacement_chars': source_text.count('\ufffd'),
                           'B_replacement_chars': full_b.count('\ufffd'), 'B_empty': not bool(full_b.strip())})
    for kind, tag in choose_source_samples(soup).items():
        source = re.sub(r'\s+', ' ', tag.get_text(' ', strip=True)).strip()
        extracted = extract_b(tag)
        source_compact = comparison_text(source)
        sample_rows.append({'company': company, 'sample': kind,
                            'same_characters_in_local_B': bool(source_compact) and source_compact == comparison_text(extracted, from_b=True),
                            'found_in_full_B': bool(source_compact) and source_compact in full_b_compact,
                            'full_B_occurrences': full_b_compact.count(source_compact),
                            'XML_sample': source[:160], 'B_sample': extracted[:160]})
    del soup, xml_bytes

display(pd.DataFrame(sample_rows))
display(pd.DataFrame(integrity_rows))
if any(not row['same_characters_in_local_B'] or not row['found_in_full_B'] for row in sample_rows):
    print('검토 필요: 표본 XML과 B 결과가 일치하지 않는 행이 있습니다.')
else:
    print('선택한 6개 XML 표본 문자열이 B 전체에도 존재합니다. 반복 문구의 원래 위치까지 확인한 것은 아닙니다.')

두 표의 `False` 또는 U+FFFD 대체 문자(`�`)가 발견되면 해당 XML 구간을 직접 확인합니다. `full_B_occurrences`가 2 이상이면 다른 위치의 같은 문구가 일치를 만든 것일 수도 있습니다. 모두 정상이어도 **6개 표본 문자열만** 대조한 것이며, 위치별 누락이나 문서 전체의 손실이 없다고 결론낼 수 없습니다.